In [1]:
import pandas as pd
import numpy as np

# 1. Carregar a base original (Chácara)

In [2]:
# Carrega o arquivo de entrada (arquivo organizado em `data/filling_Ceps/`)
df = pd.read_csv('../../data/filling_Ceps/Elvira - Chacara - INEP 35107700 Dados ADS_coords_corrigidas_com_enderecos.csv')

# 2. Tratar a renda para número

In [3]:
import re

def parse_renda_seguro(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    s = re.sub(r"[^0-9,\.]", "", s)
    if "," in s:
        s = s.replace(".", "")
    elif s.count(".") > 1:
        s = s.replace(".", "")
    s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan


df["renda_media_num"] = df["renda_media"].apply(parse_renda_seguro)


# 3. Criar Total_0_9 (0–4 + 5–9 anos)

In [4]:
# 3. Criar Total_0_18 (0-4 + 5-9 + 10-14 + 15-19 anos)
df["Total_0_18"] = (
    df["v01031_0_4anos"].fillna(0)
    + df["v01032_5_9anos"].fillna(0)
    + df["v01033_10_14anos"].fillna(0)
    + df["v01034_15_19anos"].fillna(0)
)
# Obs: faixa 15-19 inclui 19 (proxy para 15-18).


In [5]:
import math

def haversine_km(lat, lon, lat0, lon0):
    lat = np.radians(pd.to_numeric(lat, errors="coerce"))
    lon = np.radians(pd.to_numeric(lon, errors="coerce"))
    lat0 = math.radians(lat0)
    lon0 = math.radians(lon0)
    dlat = lat - lat0
    dlon = lon - lon0
    a = np.sin(dlat / 2) ** 2 + np.cos(lat0) * np.cos(lat) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c

# Coordenadas da unidade
escola_lat = -23.6336779
escola_lon = -46.7132727

# Distancia por ponto
if "latitude_centro" in df.columns and "longitude_centro" in df.columns:
    df["distancia_km"] = haversine_km(df["latitude_centro"], df["longitude_centro"], escola_lat, escola_lon)
else:
    df["distancia_km"] = np.nan

# Parametros da filtragem por robustez e tamanho das listas
min_pontos = 5
top_n = 12

# 4. Corrigir renda para 2025 (inflação)

In [6]:
inflation_factor = 1.155
df["renda_atualizada_2025"] = df["renda_media_num"] * inflation_factor

# 5. Score linha a linha (informativo, não consolidado)

In [7]:
df["score_trafego_2025"] = df["renda_atualizada_2025"] * df["Total_0_18"]

# 6. Consolidar por CEP (SEM SOMAR — usar média!)

In [8]:
df_cep = (
    df.groupby("CEP", as_index=False)
      .agg({
          "Bairro": lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0],
          "renda_atualizada_2025": "median",
          "Total_0_18": "median",
          "populacao_total": "median",
          "distancia_km": "median",
          "v01031_0_4anos": "median",
          "v01032_5_9anos": "median",
          "v01033_10_14anos": "median",
          "v01034_15_19anos": "median",
      })
)

pontos = df.groupby("CEP").size().reset_index(name="pontos")
df_cep = df_cep.merge(pontos, on="CEP", how="left")


In [9]:
# Renomear colunas para refletir que sao medianas

df_cep = df_cep.rename(columns={
    "renda_atualizada_2025": "renda_mediana_2025",
    "Total_0_18": "mediana_criancas_0_18",
    "populacao_total": "populacao_mediana",
    "distancia_km": "distancia_mediana_km",
    "v01031_0_4anos": "mediana_0_4",
    "v01032_5_9anos": "mediana_5_9",
    "v01033_10_14anos": "mediana_10_14",
    "v01034_15_19anos": "mediana_15_19",
})


# 7. Score final no nível do CEP (correto)

In [10]:
df_cep["score_trafego_2025"] = (
    df_cep["renda_mediana_2025"] * df_cep["mediana_criancas_0_18"]
)

# 8. Ranking final

In [11]:
top_ceps = df_cep.sort_values("score_trafego_2025", ascending=False)

print(top_ceps.head(15))

           CEP                            Bairro  renda_mediana_2025  \
382  04719-905  Chácara Santo Antônio (Zona Sul)         32647.68045   
585  05635-050                Jardim Monte Kemel         24449.50200   
253  04660-000                  Jardim Marajoara         29669.06250   
689  05679-050           Jardim Panorama D'Oeste         35331.79650   
739  05709-040                       Vila Suzana         36272.94825   
461  04747-140                       Santo Amaro         38211.82365   
110  04583-909                     Vila Cordeiro         28934.10135   
256  04661-000                   Jardim Umuarama         23449.15650   
261  04662-902                       Santo Amaro         28198.32015   
62   04564-900                    Cidade Monções         34184.38485   
280  04671-906                        Vila Sofia         25100.48310   
574  05634-001                Jardim Monte Kemel         22594.04070   
397  04726-160                     Vila Cruzeiro         24868.2

In [12]:
# Arredondar para 2 casas decimais (padrao monetario)
top_ceps["renda_mediana_2025"] = top_ceps["renda_mediana_2025"].round(2)
top_ceps["score_trafego_2025"] = top_ceps["score_trafego_2025"].round(2)
if "distancia_mediana_km" in top_ceps.columns:
    top_ceps["distancia_mediana_km"] = top_ceps["distancia_mediana_km"].round(2)


In [13]:
# Salvar resultado agregado na pasta do notebook (comportamento original)
top_ceps.to_csv("chacara_top_ceps_2025.csv", index=False)

In [14]:
# Filtrar por robustez (sem filtro por bairro)
top_ceps_filtrado = top_ceps[top_ceps["pontos"] >= min_pontos].copy()

top_ceps_filtrado


,CEP,Bairro,renda_mediana_2025,mediana_criancas_0_18,populacao_mediana,distancia_mediana_km,mediana_0_4,mediana_5_9,mediana_10_14,mediana_15_19,pontos,score_trafego_2025
796,05726-130,Vila Andrade,11951.80,159.0,600.0,3.06,46.0,43.0,40.0,23.0,5,1900336.42
732,05706-290,Paraíso do Morumbi,25705.64,63.0,296.0,1.26,12.5,16.5,25.0,18.0,6,1619455.29
607,05641-030,Vila Suzana,21049.40,65.0,342.0,3.04,16.0,13.0,19.0,13.0,7,1368211.09
604,05641-010,Vila Suzana,21478.15,59.0,292.0,2.82,8.0,13.0,14.0,10.0,5,1267210.79
66,04566-000,Cidade Monções,17093.82,72.0,308.0,3.51,22.0,20.0,10.0,7.5,5,1230754.69
111,04601-000,Brooklin Paulista,17583.51,66.0,270.0,3.12,13.0,16.0,19.0,18.0,5,1160511.80
40,04562-000,Brooklin Paulista,20903.71,51.0,318.0,3.95,15.5,14.0,9.5,13.5,5,1066089.20
1,04363-001,Vila Mascote,19790.19,51.0,236.0,4.52,13.5,13.0,16.0,14.0,5,1009299.48
744,05711-001,Jardim Caboré,12730.75,79.0,367.0,3.81,24.0,28.0,16.5,18.5,6,1005729.31
778,05717-270,Vila Andrade,15458.43,63.0,357.0,2.29,14.0,16.0,13.0,19.0,5,973880.94


In [15]:
top_ceps_filtrado.to_csv("chacara_top_ceps_filtrados_2025.csv", index=False)

In [16]:
# Rankings por distancia (sem videos)
top_ceps_base = top_ceps[top_ceps["pontos"] >= min_pontos].copy()
top_ceps_proximos = top_ceps_base.sort_values("distancia_mediana_km").head(top_n)
top_ceps_proximos.to_csv("chacara_top_ceps_proximos_2025.csv", index=False)

top_ceps_filtrado_proximos = top_ceps_filtrado.sort_values("distancia_mediana_km").head(top_n)
top_ceps_filtrado_proximos.to_csv("chacara_top_ceps_filtrados_proximos_2025.csv", index=False)
